# Fase 6E: Modelo Web Form (Feature Selection)

En este notebook entrenaremos el **Modelo Jerárquico** usando únicamente un subconjunto de **9 variables Gold Standard** de muy fácil recolección para un paciente o médico. 

Esto garantiza que el modelo serializado final que exportaremos para la API no dependa de las 707 variables de NHANES, sino que sea amigable y rápido para cualquier formulario web en el mundo real.

### Variables Gold Standard Seleccionadas:
1. **`RIDAGEYR`**: Edad en años.
2. **`RIAGENDR`**: Sexo.
3. **`BMXBMI`**: Índice de Masa Corporal.
4. **`BMXWAIST`**: Perímetro de Cintura (cm).
5. **`LBXSGL`**: Glucosa en Ayunas (mg/dL).
6. **`LBDSTRSI`**: Triglicéridos.
7. **`LBDHDD`**: Colesterol HDL ("Bueno").
8. **`BPXSY1`**: Presión Arterial Sistólica.
9. **`BPXDI1`**: Presión Arterial Diastólica.


In [1]:
import pandas as pd
import numpy as np
import warnings
from pathlib import Path
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import f1_score
from xgboost import XGBClassifier
from kedro.framework.session import KedroSession
from kedro.framework.startup import bootstrap_project

warnings.filterwarnings('ignore')

# Inicializar Kedro
project_path = Path("..")
bootstrap_project(project_path)
with KedroSession.create(project_path=project_path) as session:
    catalog = session.load_context().catalog
    df = catalog.load('df_master')


[06/16/26 16:49:32] INFO     Using                                                                  ]8;id=395738;file://c:\Users\alarc\AppData\Local\Programs\Python\Python312\Lib\site-packages\kedro\framework\project\__init__.py\__init__.py]8;;\:]8;id=412128;file://c:\Users\alarc\AppData\Local\Programs\Python\Python312\Lib\site-packages\kedro\framework\project\__init__.py#269\269]8;;\
                             'c:\Users\alarc\AppData\Local\Programs\Python\Python312\Lib\site-packa                
                             ges\kedro\framework\project\rich_logging.yml' as logging                              
                             configuration.                                                                        

[06/16/26 16:49:34] INFO     Kedro is sending anonymous usage data with the sole purpose of improving ]8;id=536568;file://c:\Users\alarc\AppData\Local\Programs\Python\Python312\Lib\site-packages\kedro_telemetry\plugin.py\plugin.py]8;;\:]8;id=268202;file://c:\Users\alarc\AppData\Local\Programs\Python\Python312\Lib\site-packages\kedro_telemetry\plugin.py#242\242]8;;\
                             the product. No personal data or IP addresses are stored on our side. To              
                             opt out, set the `KEDRO_DISABLE_TELEMETRY` or `DO_NOT_TRACK` environment              
                             variables, or create a `.telemetry` file in the current working                       
                             directory with the contents `consent: false`. To hide this message,                   
                             explicitly grant or deny consent. Read more at                                        
                             https://docs.kedro.org/en/stable/about/telemetry/                                     

                    INFO     Loading data from df_master (ParquetDataset)...                   ]8;id=106500;file://c:\Users\alarc\AppData\Local\Programs\Python\Python312\Lib\site-packages\kedro\io\data_catalog.py\data_catalog.py]8;;\:]8;id=767979;file://c:\Users\alarc\AppData\Local\Programs\Python\Python312\Lib\site-packages\kedro\io\data_catalog.py#1048\1048]8;;\

## 1. Filtrado de Dataset a 9 Variables

In [2]:
# Variables seleccionadas para el Web Form
web_features = [
    'RIDAGEYR', 'RIAGENDR', 'BMXBMI', 'BMXWAIST', 'LBXSGL', 
    'LBDSTRSI', 'LBDHDD', 'BPXSY1', 'BPXDI1'
]
TARGET_COL = 'DIABETES_RISK'

X = df[web_features].copy()
y = df[TARGET_COL].copy()

numeric_cols = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f"Shape del dataset Web Form: {X.shape}")
print(f"Variables seleccionadas: {web_features}")


Shape del dataset Web Form: (6045, 9)
Variables seleccionadas: ['RIDAGEYR', 'RIAGENDR', 'BMXBMI', 'BMXWAIST', 'LBXSGL', 'LBDSTRSI', 'LBDHDD', 'BPXSY1', 'BPXDI1']


## 2. Entrenamiento Jerárquico Anti-Overfitting

In [3]:
# Preprocesador adaptado solo a estas 9 variables
def build_preprocessor(num_cols, cat_cols):
    num_tf = Pipeline([
        ('imp', SimpleImputer(strategy='median')), 
        ('sc', StandardScaler())
    ])
    tfs = [('num', num_tf, num_cols)]
    if cat_cols:
        cat_tf = Pipeline([
            ('imp', SimpleImputer(strategy='most_frequent')),
            ('enc', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))
        ])
        tfs.append(('cat', cat_tf, cat_cols))
    return ColumnTransformer(transformers=tfs, remainder='drop')

y_train_m1 = (y_train == 2).astype(int)

model1_pipeline = Pipeline([
    ('preprocessor', build_preprocessor(numeric_cols, categorical_cols)),
    ('classifier', XGBClassifier(
        objective='binary:logistic', eval_metric='logloss',
        n_estimators=700, max_depth=5, learning_rate=0.01,
        min_child_weight=1, subsample=0.9, colsample_bytree=0.5,
        gamma=0.5, reg_alpha=0.01, reg_lambda=3.0,
        scale_pos_weight=(len(y_train_m1) - y_train_m1.sum()) / y_train_m1.sum(),
        random_state=42, n_jobs=-1
    ))
])

# Validacion cruzada para umbral de M1
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
y_proba_cv_m1 = cross_val_predict(model1_pipeline, X_train, y_train_m1, cv=cv, method='predict_proba')[:, 1]
thresholds = np.linspace(0.3, 0.8, 51)
f1_scores = [f1_score(y_train_m1, (y_proba_cv_m1 >= t).astype(int)) for t in thresholds]
best_th_m1 = thresholds[np.argmax(f1_scores)]

print(f"Umbral óptimo Modelo 1 (Web Form): {best_th_m1:.3f}")
model1_pipeline.fit(X_train, y_train_m1)

# Modelo 2 (Prediabetes)
mask_train = y_train < 2
X_train_m2 = X_train[mask_train]
y_train_m2 = y_train[mask_train]

model2_pipeline = Pipeline([
    ('preprocessor', build_preprocessor(numeric_cols, categorical_cols)),
    ('classifier', XGBClassifier(
        objective='binary:logistic', eval_metric='logloss',
        n_estimators=700, max_depth=5, learning_rate=0.01,
        min_child_weight=1, subsample=0.9, colsample_bytree=0.5,
        gamma=0.5, reg_alpha=0.01, reg_lambda=3.0,
        scale_pos_weight=(len(y_train_m2) - y_train_m2.sum()) / y_train_m2.sum(),
        random_state=42, n_jobs=-1
    ))
])
model2_pipeline.fit(X_train_m2, y_train_m2)
print("Modelos entrenados correctamente.")


Umbral óptimo Modelo 1 (Web Form): 0.750
Modelos entrenados correctamente.


## 3. Evaluación Clínica Final

In [4]:
prob_diabetes = model1_pipeline.predict_proba(X_test)[:, 1]
is_diabetes = (prob_diabetes >= best_th_m1).astype(int)
prob_prediabetes = model2_pipeline.predict_proba(X_test)[:, 1]

categories = []
for diabetes_flag, prob in zip(is_diabetes, prob_prediabetes):
    if diabetes_flag == 1:
        categories.append('Diabetes')
    else:
        if prob < 0.25:
            categories.append('1_Sin Riesgo')
        elif prob < 0.50:
            categories.append('2_Riesgo Leve')
        elif prob < 0.75:
            categories.append('3_Riesgo Moderado')
        else:
            categories.append('4_Riesgo Grave')

df_results = pd.DataFrame({
    'Clase Real': y_test.map({0: 'Sano', 1: 'Prediabetes', 2: 'Diabetes'}),
    'Clasificación Clínica': categories
})
crosstab = pd.crosstab(df_results['Clase Real'], df_results['Clasificación Clínica'])
ordered_cols = ['1_Sin Riesgo', '2_Riesgo Leve', '3_Riesgo Moderado', '4_Riesgo Grave', 'Diabetes']
ordered_rows = ['Sano', 'Prediabetes', 'Diabetes']
crosstab = crosstab.reindex(index=ordered_rows, columns=ordered_cols, fill_value=0)

for index, row in crosstab.iterrows():
    total = row.sum()
    print(f'\nPacientes que realmente son: {index.upper()} (Total: {total})')
    for col in ordered_cols:
        pct = (row[col] / total) * 100
        if pct > 0:
            print(f'  -> Clasificados como {col:.<25} {pct:>5.1f}% ({row[col]} pac.)')



Pacientes que realmente son: SANO (Total: 732)
  -> Clasificados como 1_Sin Riesgo.............  46.7% (342 pac.)
  -> Clasificados como 2_Riesgo Leve............  22.8% (167 pac.)
  -> Clasificados como 3_Riesgo Moderado........  23.5% (172 pac.)
  -> Clasificados como 4_Riesgo Grave...........   5.7% (42 pac.)
  -> Clasificados como Diabetes.................   1.2% (9 pac.)

Pacientes que realmente son: PREDIABETES (Total: 327)
  -> Clasificados como 1_Sin Riesgo.............   6.7% (22 pac.)
  -> Clasificados como 2_Riesgo Leve............  16.5% (54 pac.)
  -> Clasificados como 3_Riesgo Moderado........  42.8% (140 pac.)
  -> Clasificados como 4_Riesgo Grave...........  23.2% (76 pac.)
  -> Clasificados como Diabetes.................  10.7% (35 pac.)

Pacientes que realmente son: DIABETES (Total: 150)
  -> Clasificados como 1_Sin Riesgo.............   0.7% (1 pac.)
  -> Clasificados como 2_Riesgo Leve............   2.7% (4 pac.)
  -> Clasificados como 3_Riesgo Moderado........  16